# Reproduce ALL: train + honestly test all three LoRA methods (one run)

Single source of truth for the paper. On **one A100**, reproducibly (seed 42), this trains and tests:

1. **legacy / "original"** — pooled S&P 500 on a **2014+ bull window** (recreates the ~80% raw-accuracy condition, then debunks it).
2. **"split"** — **per-sector** S&P 500 adapters (11 sectors) + pooled, honest 2005+ walk-forward.
3. **"nasdaq"** — pooled NASDAQ-100, honest 2005+ walk-forward.

> **Read this first.** The ~80% was *raw* directional accuracy on a bull window with **no base-rate baseline** — the base rate, not skill. This notebook reproduces that high **raw** number and then shows **excess over always-up is about 0**. Under the honest test all three look "close" (high raw accuracy) precisely **because it is the market trend**. **High raw accuracy is NOT skill.**

Versions (frozen, checksummed): `sp500_2005-01-01_2026-01-01_f2026-06-04`, `nasdaq100_2005-01-01_2026-01-01_f2026-06-16`.

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
!nvidia-smi -L

## 1. Get the code (`main`)

In [ ]:
%cd /content
!rm -rf when-directional-accuracy-lies
!git clone --branch main https://github.com/taizhenC/when-directional-accuracy-lies.git
%cd /content/when-directional-accuracy-lies
!git log --oneline -3

## 2. Install dependencies

In [ ]:
!pip install -q git+https://github.com/google-research/timesfm.git --no-deps
!pip install -q peft accelerate yfinance pandas numpy scipy pyarrow huggingface_hub einops lxml matplotlib
# peft>=0.17 raises on Colab's pre-installed torchao; we do not use it.
!pip uninstall -y -q torchao

## 3. Hugging Face auth

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    hf = userdata.get('HF_TOKEN')
except Exception:
    hf = None
if hf:
    login(token=hf); print('HF login OK')
else:
    login()

## 4. Setup + preflight
Loads both frozen datasets (checksum-verified) and asserts the 11 SPDR sector anchors and non-empty sector coverage exist for the per-sector S&P run, so it fails *here* rather than mid-run.

In [ ]:
import sys; sys.path.insert(0, '.')
import json
import finetune.run_benchmark as rb
import scripts.render_tables as rt
from finetune.data_frozen import load_frozen
from finetune.data_collector import SECTOR_TO_ETF

SP  = 'sp500_2005-01-01_2026-01-01_f2026-06-04'
NDQ = 'nasdaq100_2005-01-01_2026-01-01_f2026-06-16'

def save_local(result_dir, tag):
    """Download result files with a tag prefix (no cross-method collision)."""
    import os, shutil
    from google.colab import files
    for name in ['raw.json', 'tables.md', 'run_meta.json']:
        src = os.path.join(result_dir, name)
        if os.path.exists(src):
            dst = tag + '_' + name; shutil.copy(src, dst); print('download ->', dst); files.download(dst)

for v in (SP, NDQ):
    recs, meta = load_frozen(v)
    print(v, '| stocks', meta['n_stocks'], '| etfs', meta['n_etfs'])

recs, meta = load_frozen(SP)
tmeta = meta['tickers']; present = set(tmeta)
missing_etf = [e for e in SECTOR_TO_ETF.values() if e not in present]
assert not missing_etf, 'missing SPDR anchors: ' + str(missing_etf)
by_sec = {}
for t, i in tmeta.items():
    if i['category'] == 'stock':
        by_sec[i['sector']] = by_sec.get(i['sector'], 0) + 1
print('S&P sector stock counts:', by_sec)
empty = [s for s in SECTOR_TO_ETF if by_sec.get(s, 0) == 0]
assert not empty, 'S&P sectors with no stocks: ' + str(empty)
print('preflight OK - all anchors present, all sectors populated')

## 5. Reproducibility gate (smoke twice, isolated)
Proves seeded determinism before the long run. Smoke writes to `results/smoke/` with `run_tag="smoke"` so it never pollutes or gets reused by the full runs.

In [ ]:
import math
def tableA_acc(p):
    return {(r['method'], r['fold'], r['split'], r['category'], r['horizon']): r.get('acc')
            for r in p['tables']['A']}
def _num(x):
    return x is not None and not (isinstance(x, float) and math.isnan(x))

p1 = rb.run(version=NDQ, pooled_only=True, smoke=True, epochs=2, out_root='results/smoke', run_tag='smoke')
p2 = rb.run(version=NDQ, pooled_only=True, smoke=True, epochs=2, out_root='results/smoke', run_tag='smoke')
a1, a2 = tableA_acc(p1), tableA_acc(p2)
shared = set(a1) & set(a2)
# random_walk makes no directional call -> acc is NaN (not None); skip non-numeric cells.
diffs = [abs(a1[k] - a2[k]) for k in shared if _num(a1[k]) and _num(a2[k])]
maxd = max(diffs) if diffs else 0.0
print('max |delta acc| = %.2e over %d numeric cells (of %d)' % (maxd, len(diffs), len(shared)))
assert maxd < 1e-3, 'NOT reproducible: ' + str(maxd)
print('OK gate PASSED - seeded training is deterministic')

## 6. Recreate-and-debunk the ~80% (legacy)
Trains pooled S&P on **2014->2023**, early-stops on **2023->2024**, scores **2024->2026** (the bull window). Expect raw accuracy climbing toward ~0.70-0.80 at h=128 with **excess about 0** - the artifact, exposed.

In [ ]:
legacy = rb.legacy_raw_eval(version=SP)   # writes results/legacy/<SP>_legacy_2014_2026/
save_local('results/legacy/' + SP + '_legacy_2014_2026', 'legacy')

## 7. Honest benchmark - S&P 500 (pooled "original" + per-sector "split")
First a fast **per-sector dry run** (all 11 sectors, 2 epochs, 1 fold, isolated) to prove the path before the multi-hour job; then the full seeded run (`resume=True` so a restart continues). Trains pooled + 11 sectors x 3 folds.

In [ ]:
# per-sector dry run (isolated): exercises all-sector training + concat + fail-loud coverage
_ = rb.run(version=SP, pooled_only=False, smoke=True, smoke_sectors='all', epochs=2,
           out_root='results/smoke', run_tag='smoke')
print('per-sector dry run OK')

# FULL S&P: pooled ("original") + per-sector ("split") + the registered per_sector-vs-pooled test
payload_sp = rb.run(version=SP, pooled_only=False, resume=True)
v = payload_sp['version']
open('results/benchmark/' + v + '/tables.md', 'w').write(rt.render(json.load(open('results/benchmark/' + v + '/raw.json'))))
save_local('results/benchmark/' + v, 'sp500')

## 8. Honest benchmark - NASDAQ-100 (pooled)

In [ ]:
payload_ndq = rb.run(version=NDQ, pooled_only=True, resume=True)
v = payload_ndq['version']
open('results/benchmark/' + v + '/tables.md', 'w').write(rt.render(json.load(open('results/benchmark/' + v + '/raw.json'))))
save_local('results/benchmark/' + v, 'nasdaq')

## 9. Unified comparison + figures
Method x universe, **raw acc AND excess** side by side (the artifact across all methods), plus the per-sector verdict and figures.

In [ ]:
import numpy as np
def cell(raw, method, field, split='held_out', cat='stock', h=128):
    vs = [r[field] for r in raw['tables']['A'] if r['method'] == method and r['split'] == split
          and r['category'] == cat and r['horizon'] == h and r.get(field) is not None]
    return float(np.mean(vs)) if vs else None

sp  = json.load(open('results/benchmark/' + payload_sp['version'] + '/raw.json'))
ndq = json.load(open('results/benchmark/' + payload_ndq['version'] + '/raw.json'))
leg = json.load(open('results/legacy/' + SP + '_legacy_2014_2026/raw.json'))

print('=== held-out stocks, h=128: RAW acc vs EXCESS over always-up ===')
print('%-32s %8s %10s %8s' % ('method', 'raw_acc', 'always_up', 'excess'))
def row(name, raw, method):
    a = cell(raw, method, 'acc'); au = cell(raw, method, 'always_up_acc'); e = cell(raw, method, 'excess_acc')
    if a is not None: print('%-32s %8.3f %10.3f %+8.3f' % (name, a, au, e))
row('legacy_sp500_pooled_2014', leg, 'legacy_pooled_2014')
row('sp500_pooled_walkforward', sp, 'pooled')
row('sp500_per_sector_walkforward', sp, 'per_sector')
row('nasdaq_pooled_walkforward', ndq, 'pooled')
print()
print('Reminder: high raw accuracy = the base rate (trend), not skill. Excess is the skill metric.')

ps = [r for r in sp['tables']['D'] if r['comparison'] == 'per_sector vs pooled'
      and r['split'] == 'held_out' and r['horizon'] == 128]
print('per_sector vs pooled (held-out, h=128) DM:', [round(r['statistic'], 2) for r in ps],
      '| primary flagged:', any(r.get('primary') for r in ps))

!python -m scripts.make_figures --out paper/figures "NASDAQ-100=results/benchmark/{payload_ndq['version']}/raw.json" "S&P 500=results/benchmark/{payload_sp['version']}/raw.json"
print('figures -> paper/figures/')

## 10. Save everything locally
All result sets are already downloaded after each part; this re-saves them together plus the figures.

In [ ]:
for d, tag in [('results/legacy/' + SP + '_legacy_2014_2026', 'legacy'),
               ('results/benchmark/' + payload_sp['version'], 'sp500'),
               ('results/benchmark/' + payload_ndq['version'], 'nasdaq')]:
    save_local(d, tag)
import shutil
from google.colab import files
shutil.make_archive('figures', 'zip', 'paper/figures'); files.download('figures.zip')